In [1]:
import os
os.chdir(r"..\models\..")

import math
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F
import pickle

from tokenizer import Tokenizer
from transformer.transformer import Transformer

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, random_split

In [2]:
# Training Tokenizer
data = "data/translation_en_es_data.csv"
df = pd.read_csv(data)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Dropping missing values >:(
train_df = train_df.dropna(subset=["English", "Spanish"])

tokenizer = Tokenizer(vocab_size=100_000)

texts = list(train_df["English"]) + list(train_df["Spanish"])
tokenizer.fit(texts)

vocab_size = len(tokenizer.word_to_idx)
print(f"Vocabulary Size: {vocab_size}")

with open("models/translator_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

Vocabulary Size: 74044


In [12]:
class TranslationDataset(Dataset):
    def __init__(self, df, tokenizer, src_max_len=128, trg_max_len=128):
        self.src_texts = df["English"].tolist()
        self.trg_texts = df["Spanish"].tolist()
        self.tokenizer = tokenizer
        self.src_max_len = src_max_len
        self.trg_max_len = trg_max_len

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        src = self.tokenizer.transform(self.src_texts[idx])
        trg = self.tokenizer.transform(self.trg_texts[idx])

        src = self.tokenizer.pad_sequence([src], self.src_max_len)[0]
        trg = self.tokenizer.pad_sequence([trg], self.trg_max_len)[0]

        return torch.tensor(src, dtype=torch.long), torch.tensor(trg, dtype=torch.long)

In [4]:
# Plotting Metrics Function
import matplotlib.pyplot as plt

def plot_metrics(train_losses, val_losses, perplexities):
    epochs = range(1, len(val_losses) + 1)

    plt.figure(figsize=(12, 5))

    # Loss Plotting
    plt.subplot(1, 3, 1)
    plt.plot(epochs, train_losses, label="Train Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss")
    plt.grid()
    plt.legend()

    # Accuracy Plotting
    plt.subplot(1, 3, 2)
    plt.plot(epochs, val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Validation Loss")
    plt.grid()
    plt.legend()
    
    # Perplexity Plotting
    plt.subplot(1, 3, 3)
    plt.plot(epochs, perplexities, label="Perplexity")
    plt.xlabel("Epoch")
    plt.ylabel("Perplexity")
    plt.title("Perplexity per Epoch")
    plt.grid()
    plt.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Training the model
train_data = TranslationDataset(train_df, tokenizer)
val_data = TranslationDataset(val_df, tokenizer)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
src_pad_idx = tokenizer.word_to_idx["<PAD>"]
trg_pad_idx = tokenizer.word_to_idx["<PAD>"]

model = Transformer(
    vocab_size=vocab_size,
    src_pad_idx=src_pad_idx,
    trg_pad_idx=trg_pad_idx,
    embedding_dim=256,
    num_layers=4,
    num_heads=8,
    d_ff=1024,
    max_len=128,
    dropout=0.1,
    device=device
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=trg_pad_idx)
optimizer = optim.Adam(model.parameters(), lr=3e-4)

def train(model, train_loader, val_loader, criterion, optimizer, device, epochs=100):

    best_val_loss = float("inf")

    train_losses = []
    val_losses = []
    perplexities = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for src, trg in train_loader:
            src = src.to(device)
            trg = trg.to(device)

            trg_input = trg[:, :-1]
            trg_target = trg[:, 1:]

            optimizer.zero_grad()
            output = model(src, trg_input)

            loss = criterion(output.reshape(-1, output.shape[-1]), trg_target.reshape(-1))
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        
        with torch.no_grad():
            src = src.to(device)
            trg = trg.to(device)

            output = model(src, trg_input)
            loss = criterion(output.reshape(-1, output.shape[-1]), trg_target.reshape(-1))
            val_loss += loss.item()

        val_loss /= len(val_loader)
        perplexity = math.exp(val_loss)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        perplexities.append(perplexity)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Perplexity: {perplexity:.2f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "models/generator_best_model.pt")

    plot_metrics(train_losses, val_losses, perplexities)

print("\nCUDA available:", torch.cuda.is_available())
print("CUDA device name:", torch.cuda.get_device_name(torch.cuda.current_device()))
train(model, train_loader, val_loader, criterion, optimizer, device, epochs=120)
